# Train Monte Carlo Prediction

Monte Carlo prediction estimates a fixed policy's state values by averaging complete sampled returns. This notebook evaluates a random policy on `FrozenLake-v1`.

## Return average

$$V(s)\leftarrow V(s)+\frac{G_t-V(s)}{N(s)}.$$

Here $G_t$ is the complete return observed after visiting state $s$, and $N(s)$ is that state's visit count.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import MonteCarloPrediction, MonteCarloPredictionConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "FrozenLake-v1"

In [ ]:
env = gym.make(ENV_ID, is_slippery=True)
policy = np.full(
    (env.observation_space.n, env.action_space.n),
    1 / env.action_space.n,
)
config = MonteCarloPredictionConfig(seed=7)

agent = MonteCarloPrediction(policy, env, config=config)
agent.learn(total_timesteps=20_000, progress_bar=False)
print(agent.values.reshape(4, 4))
print(agent.visit_counts.reshape(4, 4))
env.close()

In [ ]:
plt.imshow(agent.values.reshape(4, 4), cmap="viridis")
plt.colorbar(label="Estimated value")
plt.title(f"Random-policy state values on {ENV_ID}")
plt.show()

## Watch the evaluated policy

This opens a window and runs 5 episodes by sampling from the fixed random policy.

In [ ]:
evaluation_env = gym.make(
    ENV_ID, render_mode="human", is_slippery=True
)
try:
    result = evaluate_policy(
        agent, evaluation_env, episodes=5, deterministic=False, seed=10
    )
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")